# ARMA Parameter Recovery -- Visualization (V2 Backbone + GRU h128 l3 Head)

Loads the pre-trained ConfigurableModel (GRU+FFN4x, H1024, 12L) and the GRU recovery head
(hidden_dim=128, num_gru_layers=2), then evaluates on 300 test ARMA(4,4) processes.

**Note:** The checkpoint `recovery_p5_v2_gru_h128_l3_best.pth` was trained via
`create_recovery_head("gru", ...)` which does not forward `num_gru_layers` to `GRURecoveryHead`,
so the actual model uses the default `num_gru_layers=2` (676,488 params). The filename `l3`
reflects the CLI flag, not the effective architecture.

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from arma import generate_arma_batch
from train_contrastive_v2 import ConfigurableModel
from train_parameter_recovery import GRURecoveryHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_ARMA_PARAMS = 4
NUM_TEST_SAMPLES = 300
COEFF_NAMES = [f"AR[{i}]" for i in range(NUM_ARMA_PARAMS)] + [f"MA[{i}]" for i in range(NUM_ARMA_PARAMS)]

In [ ]:
# -- Load V2 backbone --

model = ConfigurableModel(
    C=4, H=1024, W=32,
    encoder_type="gru",
    intermediate_dim=None,
    num_layers=12,
    nhead=8,
    ffn_mult=4.0,
    dropout=0.1,
    activation="gelu",
    depthwise_conv=3,
)
model.load_state_dict(torch.load("v2_2M_model_best.pth", map_location=device))
model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print("ConfigurableModel loaded (GRU+FFN4x, H1024, 12L)")

# -- Load GRU recovery head --
# The factory bug means num_gru_layers was ignored; actual default = 2.
# Params: 676,488

param_head = GRURecoveryHead(H=1024, hidden_dim=128, num_arma_params=NUM_ARMA_PARAMS, num_gru_layers=2)
param_head.load_state_dict(
    torch.load("recovery_search_logs/recovery_p5_v2_gru_h128_l3_best.pth", map_location=device)
)
param_head = param_head.to(device)
param_head.eval()

num_params = sum(p.numel() for p in param_head.parameters())
print(f"GRU recovery head loaded (hidden=128, gru_layers=2, params={num_params:,})")

In [ ]:
# -- Helpers --

def extract_latent_features(model, x):
    """Extract per-channel latent features from ConfigurableModel."""
    with torch.no_grad():
        _, h = model(x)
        B, T, C, H = h.shape
        return h.permute(0, 2, 1, 3).reshape(B * C, T, H)


def prepare_data(batch_size=32, seed=None):
    """Generate ARMA(4,4) data and extract true AR/MA coefficients."""
    x, parameters = generate_arma_batch(
        batch_size=batch_size, T_raw=4096, C=4, seed=seed, dimension=4
    )
    x = x.to(device)
    true_ar, true_ma = [], []
    for ar_poly, ma_poly in parameters:
        ar = np.pad(-ar_poly[1:], (0, max(0, NUM_ARMA_PARAMS - len(ar_poly) + 1)))[:NUM_ARMA_PARAMS]
        ma = np.pad( ma_poly[1:], (0, max(0, NUM_ARMA_PARAMS - len(ma_poly) + 1)))[:NUM_ARMA_PARAMS]
        true_ar.append(ar)
        true_ma.append(ma)
    return (
        x,
        torch.tensor(np.array(true_ar), dtype=torch.float32, device=device),
        torch.tensor(np.array(true_ma), dtype=torch.float32, device=device),
    )

## Generate predictions on 300 test samples (seed=42)

In [ ]:
# Collect predictions and ground truth for all 300 test samples.
# Each sample generates batch_size=1 with C=4 channels, so we get
# 4 independent (true, pred) pairs per sample -> 1200 total.

all_true_ar = []
all_pred_ar = []
all_true_ma = []
all_pred_ma = []

with torch.no_grad():
    for i in range(NUM_TEST_SAMPLES):
        x, ar_true, ma_true = prepare_data(batch_size=1, seed=42 + i)
        h = extract_latent_features(model, x)           # [4, T, 1024]
        pred_ar, pred_ma = param_head(h)                 # [4, T, 4] each
        # Average over time dimension
        all_true_ar.append(ar_true.cpu().numpy())        # [4, 4]
        all_pred_ar.append(pred_ar.mean(1).cpu().numpy())
        all_true_ma.append(ma_true.cpu().numpy())
        all_pred_ma.append(pred_ma.mean(1).cpu().numpy())
        if (i + 1) % 100 == 0:
            print(f"  processed {i + 1}/{NUM_TEST_SAMPLES}")

true_ar = np.concatenate(all_true_ar)   # [1200, 4]
pred_ar = np.concatenate(all_pred_ar)
true_ma = np.concatenate(all_true_ma)
pred_ma = np.concatenate(all_pred_ma)

print(f"Collected {true_ar.shape[0]} channel-level predictions "
      f"({NUM_TEST_SAMPLES} samples x 4 channels)")

## 1. True vs Predicted bar chart (5 random samples)

In [ ]:
rng = np.random.default_rng(42)
sample_indices = rng.choice(true_ar.shape[0], size=5, replace=False)

fig, axes = plt.subplots(5, 1, figsize=(12, 18))

for ax_i, idx in enumerate(sample_indices):
    true_all = np.concatenate([true_ar[idx], true_ma[idx]])   # 8 coefficients
    pred_all = np.concatenate([pred_ar[idx], pred_ma[idx]])

    x_pos = np.arange(len(COEFF_NAMES))
    w = 0.35
    ax = axes[ax_i]
    ax.bar(x_pos - w / 2, true_all, w, label="True", color="steelblue")
    ax.bar(x_pos + w / 2, pred_all, w, label="Predicted", color="indianred")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(COEFF_NAMES)
    ax.set_ylabel("Value")
    ax.set_title(f"Sample {idx} -- 8 ARMA coefficients")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="grey", lw=0.5)

plt.tight_layout()
plt.savefig("fig_true_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Scatter plots for all 8 coefficients with regression lines and Pearson r

In [ ]:
fig, axes = plt.subplots(2, NUM_ARMA_PARAMS, figsize=(4 * NUM_ARMA_PARAMS, 8))

threshold = 0.05  # ignore near-zero true values for correlation

for j in range(NUM_ARMA_PARAMS):
    for row, (name, true, pred) in enumerate([("AR", true_ar, pred_ar), ("MA", true_ma, pred_ma)]):
        ax = axes[row, j]
        ax.scatter(true[:, j], pred[:, j], alpha=0.12, s=6, color="steelblue", edgecolors="none")

        # y=x reference line
        lims = [-1.0, 1.0]
        ax.plot(lims, lims, "r--", lw=1, label="y=x")

        # Regression line
        mask = np.abs(true[:, j]) > threshold
        if mask.sum() > 2:
            slope, intercept, r_value, _, _ = stats.linregress(true[mask, j], pred[mask, j])
            xs = np.linspace(-1, 1, 100)
            ax.plot(xs, slope * xs + intercept, color="darkorange", lw=1.2,
                    label=f"fit (r={r_value:.3f})")
            ax.set_title(f"{name}[{j}]  r={r_value:.3f}")
        else:
            ax.set_title(f"{name}[{j}]")

        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(f"True {name}[{j}]")
        ax.set_ylabel(f"Predicted {name}[{j}]")
        ax.legend(fontsize=7, loc="upper left")
        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal")

plt.tight_layout()
plt.savefig("fig_scatter_plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Error distribution histograms

In [ ]:
# Per-sample MSE (averaged over 4 channels that belong to the same sample)
ar_mse_per_channel = ((pred_ar - true_ar) ** 2).mean(axis=1)  # [1200]
ma_mse_per_channel = ((pred_ma - true_ma) ** 2).mean(axis=1)
total_mse_per_channel = ar_mse_per_channel + ma_mse_per_channel

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, data, title, color in [
    (axes[0], ar_mse_per_channel,    "AR Parameter MSE",    "steelblue"),
    (axes[1], ma_mse_per_channel,    "MA Parameter MSE",    "seagreen"),
    (axes[2], total_mse_per_channel, "Total Parameter MSE", "mediumpurple"),
]:
    ax.hist(data, bins=40, alpha=0.75, color=color, edgecolor="white", linewidth=0.5)
    mean_val = data.mean()
    median_val = np.median(data)
    ax.axvline(mean_val, color="red", ls="--", lw=1.5,
               label=f"Mean: {mean_val:.4f}")
    ax.axvline(median_val, color="orange", ls=":", lw=1.5,
               label=f"Median: {median_val:.4f}")
    ax.set_xlabel("MSE")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("fig_error_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Per-coefficient sign agreement

In [ ]:
threshold = 0.05
sign_agreements = []
labels = []

for name, true, pred in [("AR", true_ar, pred_ar), ("MA", true_ma, pred_ma)]:
    for j in range(NUM_ARMA_PARAMS):
        mask = np.abs(true[:, j]) > threshold
        if mask.sum() > 0:
            agree = (np.sign(true[mask, j]) == np.sign(pred[mask, j])).mean()
        else:
            agree = 0.0
        sign_agreements.append(agree)
        labels.append(f"{name}[{j}]")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue"] * NUM_ARMA_PARAMS + ["seagreen"] * NUM_ARMA_PARAMS
bars = ax.bar(labels, sign_agreements, color=colors, edgecolor="white", linewidth=0.8)

# Annotate values
for bar, val in zip(bars, sign_agreements):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=10)

ax.set_ylabel("Sign Agreement")
ax.set_title(f"Per-coefficient Sign Agreement (|true| > {threshold})")
ax.set_ylim(0, 1.08)
ax.axhline(0.5, color="grey", ls=":", lw=1, label="chance (50%)")
ax.grid(True, alpha=0.2, axis="y")
ax.legend()

plt.tight_layout()
plt.savefig("fig_sign_agreement.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Summary metrics

In [ ]:
print("=" * 65)
print("  Parameter Recovery Performance  --  GRU h128 (V2 backbone)")
print("=" * 65)

# Overall MSE
mean_ar_mse = ((pred_ar - true_ar) ** 2).mean()
mean_ma_mse = ((pred_ma - true_ma) ** 2).mean()
mean_total_mse = mean_ar_mse + mean_ma_mse

# Baseline: predicting zeros
baseline_ar = (true_ar ** 2).mean()
baseline_ma = (true_ma ** 2).mean()
baseline_total = baseline_ar + baseline_ma

print(f"\nMean AR MSE:     {mean_ar_mse:.6f}")
print(f"Mean MA MSE:     {mean_ma_mse:.6f}")
print(f"Mean Total MSE:  {mean_total_mse:.6f}")
print(f"Baseline (zero): {baseline_total:.6f}")
print(f"Improvement:     {baseline_total / mean_total_mse:.2f}x")

# Per-coefficient table
print(f"\n{'Coeff':>6s}  {'MSE':>8s}  {'MAE':>8s}  {'Pearson r':>10s}  {'Sign Agr.':>10s}  {'N (|t|>0.05)':>13s}")
print("-" * 65)
for name, true, pred in [("AR", true_ar, pred_ar), ("MA", true_ma, pred_ma)]:
    for j in range(NUM_ARMA_PARAMS):
        mse_j = ((true[:, j] - pred[:, j]) ** 2).mean()
        mae_j = np.abs(true[:, j] - pred[:, j]).mean()
        mask = np.abs(true[:, j]) > threshold
        n_valid = mask.sum()
        if n_valid > 1:
            r_j = np.corrcoef(true[mask, j], pred[mask, j])[0, 1]
            sign_j = (np.sign(true[mask, j]) == np.sign(pred[mask, j])).mean()
        else:
            r_j = float("nan")
            sign_j = float("nan")
        print(f"{name}[{j}]   {mse_j:8.5f}  {mae_j:8.5f}  {r_j:10.4f}  {sign_j:10.1%}  {n_valid:>13d}")

# Overall sign agreement
for name, true, pred in [("AR", true_ar, pred_ar), ("MA", true_ma, pred_ma)]:
    mask = np.abs(true) > threshold
    sign_all = (np.sign(true[mask]) == np.sign(pred[mask])).mean()
    r_all = np.corrcoef(true[mask], pred[mask])[0, 1]
    print(f"{name} ALL            {r_all:10.4f}  {sign_all:10.1%}")

print("\n" + "=" * 65)